In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install sentence-transformers sacrebleu transformers -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("LaBSE")

# ── Dataset: word + multiple contexts + human translations ────────────────────
# Format: {word: {source_lang, target_lang, contexts: [(src_sentence, tgt_sentence)]}}
# Contexts should be diverse — different meanings/uses of the same word.
# The key insight: a GOOD translation should be close in embedding space
# ACROSS ALL contexts, not just the most common one.

DATA = {
    "bank": {
        "source_lang": "en",
        "target_lang": "es",
        "contexts": [
            ("I deposited money at the bank.",              "Deposité dinero en el banco."),
            ("She sat on the bank of the river.",           "Ella se sentó en la orilla del río."),
            ("The blood bank needs more donations.",        "El banco de sangre necesita más donaciones."),
            ("He robbed the bank at noon.",                 "Él robó el banco al mediodía."),
            ("The plane made a sharp bank to the left.",    "El avión hizo un viraje brusco a la izquierda."),
            ("They bank on good weather for the harvest.",  "Confían en el buen tiempo para la cosecha."),
        ],
        # Deliberately bad translations for contrast:
        "bad_translations": [
            ("I deposited money at the bank.",              "Deposité dinero en la orilla."),      # wrong sense
            ("She sat on the bank of the river.",           "Ella se sentó en el banco."),         # wrong sense
            ("The blood bank needs more donations.",        "La orilla de sangre necesita donaciones."),
            ("He robbed the bank at noon.",                 "Él robó la orilla al mediodía."),
            ("The plane made a sharp bank to the left.",    "El banco hizo un giro brusco."),
            ("They bank on good weather for the harvest.",  "Bancan el buen tiempo para la cosecha."),
        ],
    },
    "light": {
        "source_lang": "en",
        "target_lang": "fr",
        "contexts": [
            ("Turn on the light, it's dark.",               "Allume la lumière, il fait sombre."),
            ("She has a light touch on the piano.",         "Elle a un toucher léger sur le piano."),
            ("The light at the end of the tunnel.",         "La lumière au bout du tunnel."),
            ("He ordered a light beer.",                    "Il a commandé une bière légère."),
            ("Light the candle carefully.",                 "Allume la bougie avec précaution."),
            ("The painting uses light and shadow.",         "La peinture utilise la lumière et l'ombre."),
        ],
        "bad_translations": [
            ("Turn on the light, it's dark.",               "Allume le léger, il fait sombre."),
            ("She has a light touch on the piano.",         "Elle a une lumière sur le piano."),
            ("The light at the end of the tunnel.",         "Le léger au bout du tunnel."),
            ("He ordered a light beer.",                    "Il a commandé une lumière de bière."),
            ("Light the candle carefully.",                 "Léger la bougie avec précaution."),
            ("The painting uses light and shadow.",         "La peinture utilise le léger et l'ombre."),
        ],
    },
    "saudade": {
        "source_lang": "pt",
        "target_lang": "en",
        "contexts": [
            ("Tenho saudade dos meus avós.",                "I miss my grandparents."),
            ("A saudade é um sentimento tipicamente português.", "Saudade is a typically Portuguese feeling."),
            ("Ela olhou com saudade para a foto antiga.",   "She looked with longing at the old photo."),
            ("A música evocou uma saudade profunda.",       "The music evoked a deep sense of longing."),
            ("Saudade da infância nunca passa.",            "Nostalgia for childhood never fades."),
        ],
        "bad_translations": [
            ("Tenho saudade dos meus avós.",                "I am sad about my grandparents."),
            ("A saudade é um sentimento tipicamente português.", "Sadness is a typically Portuguese feeling."),
            ("Ela olhou com saudade para a foto antiga.",   "She looked sadly at the old photo."),
            ("A música evocou uma saudade profunda.",       "The music evoked deep sadness."),
            ("Saudade da infância nunca passa.",            "Childhood sadness never fades."),
        ],
    },
}

## TVS Algorithnm

In [ ]:
def compute_tvs(src_sentences, tgt_sentences, model, verbose=False):
    """
    Translation Validity Score.
    
    Core idea: a translation is valid not just if it's close in embedding space
    on average, but if it's close ACROSS DIVERSE contexts.
    A translation that only works in one context but fails in others should score low.
    
    Algorithm:
    1. Embed source sentences and translated sentences with LaBSE
    2. Per-context similarity = cosine(src_emb_i, tgt_emb_i)
    3. Context diversity weight = how different each context is from the mean
       (rare/unusual contexts get MORE weight — they're the hard cases)
    4. TVS = diversity-weighted mean similarity
    """
    src_embs = model.encode(src_sentences, normalize_embeddings=True)
    tgt_embs = model.encode(tgt_sentences, normalize_embeddings=True)
    
    # Per-context similarity
    per_context_sim = np.array([
        cosine_similarity(src_embs[i:i+1], tgt_embs[i:i+1])[0][0]
        for i in range(len(src_sentences))
    ])
    
    # Context diversity weights:
    # embed source sentences, find how far each is from the mean embedding
    # (unusual context = more diagnostic = higher weight)
    src_centroid = src_embs.mean(axis=0)
    diversity_scores = np.array([
        1 - cosine_similarity(src_embs[i:i+1], src_centroid.reshape(1,-1))[0][0]
        for i in range(len(src_sentences))
    ])
    # Normalize weights (add small floor so no context is ignored entirely)
    weights = diversity_scores + 0.1
    weights /= weights.sum()
    
    tvs = float(np.average(per_context_sim, weights=weights))
    unweighted = float(per_context_sim.mean())
    
    if verbose:
        for i, (sim, w, src, tgt) in enumerate(zip(per_context_sim, weights, src_sentences, tgt_sentences)):
            print(f"  [{i+1}] sim={sim:.3f}  weight={w:.3f}  | {src[:45]:<45} → {tgt[:45]}")
    
    return {
        "tvs":         round(tvs, 4),
        "unweighted":  round(unweighted, 4),
        "per_context": per_context_sim.round(4).tolist(),
        "weights":     weights.round(4).tolist(),
        "min_context": round(float(per_context_sim.min()), 4),  # worst context
        "std_context": round(float(per_context_sim.std()), 4),  # consistency
    }

# ── Run on all words ──────────────────────────────────────────────────────────
all_results = {}
for word, data in DATA.items():
    srcs = [c[0] for c in data["contexts"]]
    good = [c[1] for c in data["contexts"]]
    bad  = [c[1] for c in data["bad_translations"]]
    
    print(f"\n{'='*60}")
    print(f"Word: '{word}'  ({data['source_lang']} → {data['target_lang']})")
    print("\n  GOOD translations:")
    good_scores = compute_tvs(srcs, good, model, verbose=True)
    print(f"  TVS = {good_scores['tvs']:.4f}  (unweighted = {good_scores['unweighted']:.4f})")
    
    print("\n  BAD translations (wrong sense):")
    bad_scores  = compute_tvs(srcs, bad,  model, verbose=True)
    print(f"  TVS = {bad_scores['tvs']:.4f}  (unweighted = {bad_scores['unweighted']:.4f})")
    
    all_results[word] = {"good": good_scores, "bad": bad_scores}

## Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: TVS good vs bad per word ─────────────────────────────────────────
ax = axes[0]
words = list(all_results.keys())
good_tvs = [all_results[w]["good"]["tvs"] for w in words]
bad_tvs  = [all_results[w]["bad"]["tvs"]  for w in words]
x = np.arange(len(words))
ax.bar(x - 0.2, good_tvs, 0.35, label="Correct translation", color="#1D9E75")
ax.bar(x + 0.2, bad_tvs,  0.35, label="Wrong-sense translation", color="#E24B4A")
ax.set_xticks(x); ax.set_xticklabels(words)
ax.set_ylabel("Translation Validity Score"); ax.set_ylim(0, 1.05)
ax.set_title("TVS correctly penalises wrong-sense translations")
ax.legend(); ax.grid(axis="y", alpha=0.3)

# ── Plot 2: Per-context breakdown for 'bank' ─────────────────────────────────
ax = axes[1]
w = "bank"
n = len(all_results[w]["good"]["per_context"])
x2 = np.arange(n)
ax.bar(x2 - 0.2, all_results[w]["good"]["per_context"], 0.35,
       label="Correct", color="#1D9E75")
ax.bar(x2 + 0.2, all_results[w]["bad"]["per_context"],  0.35,
       label="Wrong sense", color="#E24B4A")
ax.step(x2, all_results[w]["good"]["weights"] * n * 0.6,
        where="mid", color="gray", linestyle="--", label="Diversity weight")
ax.set_xticks(x2)
ax.set_xticklabels([f"ctx {i+1}" for i in x2])
ax.set_ylabel("Cosine similarity"); ax.set_ylim(0, 1.05)
ax.set_title(f"'{w}' — per-context similarity\n(dashed = diversity weight: rare contexts matter more)")
ax.legend(); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("translation_validity_score.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nKey result: TVS should be clearly higher for correct translations,")
print("especially for polysemous words like 'bank' and culturally-loaded ones like 'saudade'.")